# Historical judge batch/resume supervisor
Source-only, credentials removed. Attach the original private input snapshot and checkpoints. Requires Kaggle Secrets DEEPSEEK_JUDGE_TEAM_2026 and ALIBABA_BASE_URL. Expected counts and caps belong to this historical job, not a new run. Do not run alongside an active worker.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os, sys, subprocess, json, time, signal, shutil, hashlib
from pathlib import Path
import pandas as pd
from kaggle_secrets import UserSecretsClient
root = Path('/kaggle/working/judge_full')
root.mkdir(exist_ok=True)
inputs = list(Path('/kaggle/input').rglob('scripts/judge/so_cap_du_lieu.py'))
assert len(inputs) == 1, f'Expected one input dataset, found {len(inputs)}'
shutil.copytree(inputs[0].parents[2], root, dirs_exist_ok=True)
os.chdir(root)
key = UserSecretsClient().get_secret('DEEPSEEK_JUDGE_TEAM_2026')
assert key and key.startswith('sk-')
env = os.environ.copy()
env['DASHSCOPE_API_KEY'] = key
env['ALIBABA_BASE_URL'] = UserSecretsClient().get_secret('ALIBABA_BASE_URL')
env['OPENBLAS_NUM_THREADS'] = '1'
del key
meta = root / 'data/processed/so_cap'
meta.mkdir(parents=True, exist_ok=True)
logroot = Path('/kaggle/working/judge_private_logs')
logroot.mkdir(exist_ok=True)

def run(name, args, deadline=None):
    print('[judge-phase]', name, flush=True)
    with (logroot / (name + '.log')).open('w', encoding='utf8') as f:
        proc = subprocess.Popen([sys.executable, '-I', '-X', 'utf8'] + args, env=env, stdout=f, stderr=subprocess.STDOUT, start_new_session=True)
        while proc.poll() is None:
            checkpoint = meta / ('ket_qua_lon_' + name + '.jsonl')
            if name in ('pair', 'cum', 'title') and checkpoint.exists():
                latest = {}
                with checkpoint.open(encoding='utf8') as cf:
                    for line in cf:
                        try:
                            record = json.loads(line)
                            latest[record['item_id']] = record.get('trang_thai')
                        except (ValueError, KeyError):
                            pass
                print('[judge-progress]', name, 'ok', sum((v == 'ok' for v in latest.values())), 'error', sum((v != 'ok' for v in latest.values())), flush=True)
            if deadline and time.monotonic() > deadline:
                os.killpg(proc.pid, signal.SIGINT)
                try:
                    proc.wait(timeout=90)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGTERM)
                    proc.wait(timeout=20)
                print('[judge-stopped] time_limit_checkpoint_saved', flush=True)
                return False
            time.sleep(20)
        if proc.returncode:
            print('[judge-stopped]', name, 'exit', proc.returncode, flush=True)
            return False
    return True
if not run('build', ['scripts/judge/so_cap_du_lieu.py']):
    raise RuntimeError('Index build failed')
expected = {'muc_title': (25374, '7e13d6e619eac883ebe8b53fe9595112'), 'muc_cum_ctx': (17170, '343b05872c074f3d901e8d3f326bbdb8'), 'muc_cum_lai': (4446, 'cc266bfc5394359e556599e6de3c687e'), 'muc_pair': (34534, '3485e426dfe3409f650f1c4708adb594')}
for name, (n, h) in expected.items():
    d = pd.read_parquet(meta / (name + '.parquet'))
    assert len(d) == n and hashlib.md5('\n'.join(sorted(d.item_id.astype(str))).encode()).hexdigest() == h, name
print('[judge-health] input_verified', flush=True)
deadline = time.monotonic() + 8 * 3600 + 30 * 60
pipeline_ok = True
for kind, loai, cap, n in [('pair', 'pair', 1400000, 5000), ('cum', 'cum_lai', 3300000, 4446), ('title', 'title', 7500000, 25374)]:
    args = ['scripts/judge/so_cap_chon.py', '--loai', loai, '--tat-ca', '--rubric', 'v2', '--che-do', 'mot_luot', '--gom', '15', '--tat-suy-luan', '--song-song', '8', '--tran-token', str(cap), '--nha-cung-cap', 'alibaba', '--ten', 'lon_' + kind]
    if kind == 'pair':
        args += ['--top', '5000']
    if not run(kind, args, deadline):
        pipeline_ok = False
        break
    summary = json.loads((meta / ('tom_tat_lon_' + kind + '.json')).read_text(encoding='utf8'))
    path = meta / ('ket_qua_lon_' + kind + '.jsonl')
    last = {}
    for line in path.read_text(encoding='utf8').splitlines():
        item = json.loads(line)
        last[item['item_id']] = item
    ok = sum((x.get('trang_thai') == 'ok' for x in last.values()))
    errors = len(last) - ok
    print('[judge-count]', kind, 'ok', ok, 'expected', n, 'errors', errors, 'input_tokens', summary.get('tokens_vao'), 'output_tokens', summary.get('tokens_ra'), flush=True)
    if not (ok == n and errors == 0 and (not summary.get('dung_do'))):
        print('[judge-stopped]', kind, 'incomplete_checkpoint_saved', flush=True)
        pipeline_ok = False
        break
if pipeline_ok:
    if not run('merge', ['scripts/judge/ghep_bo.py', '--title', 'lon_title', '--cum', 'lon_cum', '--pair', 'lon_pair', '--cum-che-do', 'lai', '--khi-tuong-duong', 'giu_hien_tai'], deadline):
        pipeline_ok = False
    if pipeline_ok:
        try:
            result = root / 'data/processed/bo_chon'
            a = pd.read_parquet(result / 'bo_cum_chon_deepseek.parquet')
            b = pd.read_parquet(result / 'bo_pair_chon_deepseek.parquet')
            assert len(a) == len(b) == 25575 and len(a.columns) == len(b.columns) == 17
            assert a[['product_id', 'source']].equals(b[['product_id', 'source']])
            for method, frame in [('cum', a), ('pair', b)]:
                orig = pd.read_parquet(root / 'data/processed/ft_clean' / ('ft_buoc4_giu.parquet' if method == 'cum' else 'ft_buoc4_giu_pair.parquet'))
                assert frame.product_id.astype(str).tolist() == orig.product_id.astype(str).tolist()
                for source, columns in [('1688', ['title_zh', 'description_zh']), ('tiki', ['title_vi', 'description_vi'])]:
                    mask = frame.source.eq(source)
                    for col in columns:
                        assert frame.loc[mask, col].astype('string').tolist() == orig.loc[mask, col].astype('string').tolist(), (source, col)
            print('[judge-complete] both datasets verified rows=25575 columns=17', flush=True)
        except Exception as exc:
            print('[judge-stopped] output_validation_failed', type(exc).__name__, flush=True)
            pipeline_ok = False
(Path('/kaggle/working') / 'judge_outcome.json').write_text(json.dumps({'phase': 'completed' if pipeline_ok else 'resumable_stopped', 'model': 'deepseek-v4.1-flash'}), encoding='utf8')
